# LSTM Autoencoder for Anomaly Detection

## Objective
Implement deep learning-based anomaly detection using autoencoders:
1. **Deep (Dense) Autoencoder** — Baseline reconstruction-based detector
2. **LSTM Autoencoder** — Captures temporal dependencies in sequential data

Both models learn to reconstruct normal patterns. Anomalies produce higher reconstruction error (MSE), enabling threshold-based detection.

### Theoretical Foundation
Given input $\mathbf{x}$, the autoencoder learns encoder $f_\theta$ and decoder $g_\phi$ such that:
$$\mathcal{L} = \|\mathbf{x} - g_\phi(f_\theta(\mathbf{x}))\|^2$$

Anomaly score: $s(\mathbf{x}) = \|\mathbf{x} - \hat{\mathbf{x}}\|^2$ where $\hat{\mathbf{x}} = g_\phi(f_\theta(\mathbf{x}))$

In [ ]:
import sys, os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

sys.path.insert(0, os.path.abspath('..'))

from src.autoencoder import (
    build_deep_autoencoder,
    build_lstm_autoencoder,
    create_sequences,
    train_autoencoder,
    compute_reconstruction_error,
    find_optimal_threshold,
)
from src.evaluation import (
    compute_metrics,
    plot_reconstruction_error_distribution,
    print_classification_report,
)
from src.data_loader import split_data

DATA_DIR = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results', 'figures')
print('Libraries loaded')

## 6.1 Prepare Data

In [ ]:
plant_df = pd.read_csv(os.path.join(DATA_DIR, 'plant_engineered.csv'))

with open(os.path.join(DATA_DIR, 'engineered_feature_names.json'), 'r') as f:
    feat_names = json.load(f)

# Use original + power quality features (not rolling, to avoid data leakage in dense AE)
original_feats = feat_names['original']
feature_cols = [c for c in original_feats if c in plant_df.columns]

plant_clean = plant_df.dropna(subset=feature_cols).reset_index(drop=True)

# Split
X_train_full, X_test, y_train_full, y_test = split_data(plant_clean, feature_cols)

# Further split train into train + validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, stratify=y_train_full, random_state=42
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Normal-only training data
X_train_normal = X_train_scaled[y_train.values == 0]
X_val_normal = X_val_scaled[y_val.values == 0]

print(f'Training (normal only): {X_train_normal.shape}')
print(f'Validation (normal only): {X_val_normal.shape}')
print(f'Test (all): {X_test_scaled.shape}')
print(f'Features: {len(feature_cols)}')

## 6.2 Deep (Dense) Autoencoder

In [ ]:
# Build and train
dae = build_deep_autoencoder(n_features=len(feature_cols), encoding_dims=[64, 32, 16])
dae.summary()

In [ ]:
dae, dae_history = train_autoencoder(
    dae, X_train_normal, X_val_normal,
    epochs=100, batch_size=64, patience=15
)

In [ ]:
# Training curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dae_history.history['loss'], label='Train Loss')
if 'val_loss' in dae_history.history:
    ax.plot(dae_history.history['val_loss'], label='Validation Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Deep Autoencoder — Training Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'dae_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compute reconstruction errors
errors_train_normal = compute_reconstruction_error(dae, X_train_normal)
errors_test = compute_reconstruction_error(dae, X_test_scaled)

errors_test_normal = errors_test[y_test.values == 0]
errors_test_anomaly = errors_test[y_test.values == 1]

# Find optimal threshold
threshold_dae = find_optimal_threshold(errors_train_normal, errors_test, y_test.values, method='f1')

In [ ]:
# Error distribution
plot_reconstruction_error_distribution(
    errors_test_normal, errors_test_anomaly, threshold_dae,
    save_path=os.path.join(RESULTS_DIR, 'dae_error_distribution.png')
)

In [ ]:
# Evaluate
y_pred_dae = (errors_test > threshold_dae).astype(int)
dae_metrics = compute_metrics(y_test.values, y_pred_dae, errors_test)
print('\nDeep Autoencoder Results:')
for k, v in dae_metrics.items():
    print(f'  {k}: {v:.4f}' if v is not None else f'  {k}: N/A')
print_classification_report(y_test.values, y_pred_dae, title='Deep Autoencoder')

## 6.3 LSTM Autoencoder

The LSTM variant captures temporal dependencies by processing the data as sequences. This is more powerful for detecting anomalies that manifest as temporal patterns rather than point-wise deviations.

In [ ]:
# Create sequences
SEQ_LEN = 20
X_train_seq = create_sequences(X_train_normal, sequence_length=SEQ_LEN)
X_val_seq = create_sequences(X_val_normal, sequence_length=SEQ_LEN)
X_test_seq = create_sequences(X_test_scaled, sequence_length=SEQ_LEN)

print(f'Train sequences: {X_train_seq.shape}')
print(f'Val sequences: {X_val_seq.shape}')
print(f'Test sequences: {X_test_seq.shape}')

In [ ]:
lstm_ae = build_lstm_autoencoder(
    n_features=len(feature_cols),
    sequence_length=SEQ_LEN,
    latent_dim=32,
    dropout_rate=0.2
)
lstm_ae.summary()

In [ ]:
lstm_ae, lstm_history = train_autoencoder(
    lstm_ae, X_train_seq, X_val_seq,
    epochs=80, batch_size=32, patience=12
)

In [ ]:
# Training curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lstm_history.history['loss'], label='Train Loss')
if 'val_loss' in lstm_history.history:
    ax.plot(lstm_history.history['val_loss'], label='Validation Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('LSTM Autoencoder — Training Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'lstm_ae_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compute errors for LSTM
errors_lstm_test = compute_reconstruction_error(lstm_ae, X_test_seq)

# Align labels with sequences (sequences are shorter due to windowing)
y_test_seq = y_test.values[SEQ_LEN - 1:]

errors_lstm_normal = errors_lstm_test[y_test_seq == 0]
errors_lstm_anomaly = errors_lstm_test[y_test_seq == 1]

threshold_lstm = find_optimal_threshold(
    compute_reconstruction_error(lstm_ae, X_train_seq),
    errors_lstm_test, y_test_seq, method='f1'
)

In [ ]:
plot_reconstruction_error_distribution(
    errors_lstm_normal, errors_lstm_anomaly, threshold_lstm,
    save_path=os.path.join(RESULTS_DIR, 'lstm_ae_error_distribution.png')
)

In [ ]:
y_pred_lstm = (errors_lstm_test > threshold_lstm).astype(int)
lstm_metrics = compute_metrics(y_test_seq, y_pred_lstm, errors_lstm_test)
print('\nLSTM Autoencoder Results:')
for k, v in lstm_metrics.items():
    print(f'  {k}: {v:.4f}' if v is not None else f'  {k}: N/A')
print_classification_report(y_test_seq, y_pred_lstm, title='LSTM Autoencoder')

## 6.4 Save Autoencoder Results

In [ ]:
ae_results = pd.DataFrame([
    {**dae_metrics, 'model': 'Deep Autoencoder', 'type': 'autoencoder'},
    {**lstm_metrics, 'model': 'LSTM Autoencoder', 'type': 'autoencoder'},
]).set_index('model')

ae_results.to_csv(os.path.join(DATA_DIR, 'autoencoder_results.csv'))
print('Saved autoencoder results')
print(ae_results[['precision', 'recall', 'f1_score', 'mcc', 'roc_auc']].round(4).to_string())

## Summary

Both autoencoders learn to reconstruct normal operating patterns. The LSTM variant captures temporal structure, while the Dense variant provides a simpler baseline. Anomaly detection is performed via reconstruction error thresholding.